# 6단계 (권장 티어): 상품 단위 전역 다계열 딥러닝 시계열 (패션)

**지침(docs/05_06단계_가이드.md, 6단계 3-1절)**: 단위2(상품 단위)는 series가 많고(패션 상품 3,643개) series당
길이가 대부분 짧아서, 개별로 모델을 하나씩 학습시키면 과적합만 난다. **여러 상품을 한 모델에 같이 넣어
학습(전역학습, Ch.19)**시켜야 한다.

**대상**: 리뷰 30건 이상인 상품(패션 332개, 전체 9.1%, 지침 3-1절 표 기준) 중에서도, 최소 학습 샘플을
하나는 뽑을 수 있는 관측기간(input_chunk_length + 2*output_chunk_length개월) 이상인 것만 사용한다.

**검증 방침**: 처음부터 단일 hold-out이 아니라 **롤링 백테스트**(`historical_forecasts`, `retrain=False`)로
진행한다. naive baseline을 항상 포함해서 비교하고, 상품별 스케일링은 학습구간(train)의 min/max만 사용해서
검증구간 정보가 새어들어가지 않게 한다(leakage 방지, 조님 06c 방식 참고).

**모델**: `DLinearModel`(Ch.16, 가벼운 선형 baseline) vs `NHiTSModel`(Ch.13) + `QuantileRegression`
확률적 예측(Ch.18) + 그 달 실제 리뷰수를 past_covariate로 추가.

## 1. 상품 단위 시계열 구성

In [1]:
import pandas as pd
import numpy as np

reviews = pd.read_parquet("../data/processed/reviews.parquet")
fashion = reviews[(reviews["Domain"] == "패션") & reviews["RDate_parsed"].notna()].copy()
print(f"패션 도메인 RDate 있는 리뷰: {len(fashion)}건")
print(f"패션 고유 상품 수: {fashion['ProductName'].nunique()}개")

counts_by_product = fashion.groupby("ProductName").size()
MIN_REVIEWS = 30  # 팀 공지 기준 (docs/05_06단계_가이드.md 3-1절)
qualifying = counts_by_product[counts_by_product >= MIN_REVIEWS].index
print(f"리뷰 {MIN_REVIEWS}건 이상 상품: {len(qualifying)}개 / {fashion['ProductName'].nunique()}개 ({len(qualifying)/fashion['ProductName'].nunique()*100:.1f}%)")

패션 도메인 RDate 있는 리뷰: 40398건
패션 고유 상품 수: 3643개
리뷰 30건 이상 상품: 332개 / 3643개 (9.1%)


In [2]:
from darts import TimeSeries

INPUT_LEN = 6
OUTPUT_LEN = 2
# darts가 학습 샘플을 최소 1개는 뽑으려면 INPUT_LEN+OUTPUT_LEN개월이 있어야 하고, 거기에 검증용으로
# 뗄 마지막 OUTPUT_LEN개월을 더해야 한다 (조님 06b에서 실제 겪은 ValueError -- 9로 뒀다가 학습 실패).
MIN_MONTHS = INPUT_LEN + 2 * OUTPUT_LEN

def build_product_series(sub: pd.DataFrame):
    names, series_list, count_series_list = [], [], []
    skipped_short = 0
    for name, g in sub.groupby("ProductName"):
        m = g.groupby(g["RDate_parsed"].dt.to_period("M").dt.to_timestamp()).agg(
            total=("review_id", "count"),
            negative=("GeneralPolarity", lambda s: (s == -1).sum()),
        )
        m["neg_ratio"] = m["negative"] / m["total"]
        full_idx = pd.date_range(m.index.min(), m.index.max(), freq="MS")
        if len(full_idx) < MIN_MONTHS:
            skipped_short += 1
            continue
        ratio = m["neg_ratio"].reindex(full_idx).interpolate(limit_direction="both")
        counts = m["total"].reindex(full_idx).fillna(0)

        names.append(name)
        series_list.append(TimeSeries.from_times_and_values(full_idx, ratio.values))
        count_series_list.append(TimeSeries.from_times_and_values(full_idx, counts.values))

    print(f"리뷰 {MIN_REVIEWS}건 이상 상품 {len(qualifying)}개 중, 관측기간 {MIN_MONTHS}개월 이상인 "
          f"{len(series_list)}개 사용 (짧아서 제외 {skipped_short}개)")
    lengths = [len(s) for s in series_list]
    print(f"시리즈 길이: 최소 {min(lengths)}, 중앙값 {int(np.median(lengths))}, 최대 {max(lengths)}")
    return names, series_list, count_series_list

sub = fashion[fashion["ProductName"].isin(qualifying)].copy()
names, series_list, count_series_list = build_product_series(sub)

리뷰 30건 이상 상품 332개 중, 관측기간 10개월 이상인 101개 사용 (짧아서 제외 231개)
시리즈 길이: 최소 10, 중앙값 15, 최대 56


## 2. 리크리지 안전한 스케일링

상품마다 부정비율의 기준선이 다 달라서(어떤 상품은 5%대, 어떤 상품은 60%대) 스케일을 맞춰야 전역학습에서
신경망이 절대값이 아니라 상대적인 변화를 배울 수 있다. **darts의 자동 `Scaler`는 여러 시리즈를 한 리스트로
fit하면 순서에 의존해서 롤링 백테스트 구조와 맞춰 쓰기 까다롭기도 하고, 검증구간 값까지 스케일 계산에
들어가면 정보 누설이 된다.** 그래서 상품별로 **학습구간(앞 75%)의 min/max만** 직접 계산해서 수동으로
스케일링한다(검증구간 값은 스케일 계산에 안 씀 -- 조님 06c에서 검증된 방식).

In [3]:
TRAIN_FRAC = 0.75

def manual_scale(ts, lo, hi):
    return TimeSeries.from_times_and_values(ts.time_index, (ts.values() - lo) / (hi - lo + 1e-9))

# 학습구간(cut)이 최소 INPUT_LEN+OUTPUT_LEN개월은 돼야 darts가 학습 샘플을 하나라도 뽑을 수 있다.
# MIN_MONTHS=10(관측기간 기준)으로 최대한 상품을 확보했지만, 그 중 학습구간(75%)이 너무 짧아
# 실제로는 학습이 안 되는 경우가 남아있을 수 있어 여기서 한 번 더 걸러낸다.
MIN_TRAIN_LEN = INPUT_LEN + OUTPUT_LEN

train_mins, train_maxs, series_scaled, cov_scaled, train_target = [], [], [], [], []
kept_names, kept_series_list, kept_count_series_list = [], [], []
skipped_train_too_short = 0
for name, s, c in zip(names, series_list, count_series_list):
    cut = int(len(s) * TRAIN_FRAC)
    if cut < MIN_TRAIN_LEN or (len(s) - cut) < OUTPUT_LEN:
        skipped_train_too_short += 1
        continue
    tmin, tmax = float(s[:cut].values().min()), float(s[:cut].values().max())
    cmin, cmax = float(c[:cut].values().min()), float(c[:cut].values().max())
    train_mins.append(tmin); train_maxs.append(tmax)
    series_scaled.append(manual_scale(s, tmin, tmax))
    cov_scaled.append(manual_scale(c, cmin, cmax))
    train_target.append(manual_scale(s, tmin, tmax)[:cut])
    kept_names.append(name)
    kept_series_list.append(s)
    kept_count_series_list.append(c)

names, series_list, count_series_list = kept_names, kept_series_list, kept_count_series_list
print(f"학습구간이 너무 짧아 추가로 제외: {skipped_train_too_short}개")
print(f"최종 사용 상품: {len(series_scaled)}개")

학습구간이 너무 짧아 추가로 제외: 16개
최종 사용 상품: 85개


## 3. 모델 학습 + 롤링 백테스트 (retrain=False, 처음부터 백테스트 방식)

- **DLinear**: 가벼운 선형 신경망, covariate 없이 순수 baseline
- **NHiTS + 확률적 예측(QuantileRegression) + 리뷰수 covariate**: 그 달 실제 리뷰 개수를 `past_covariates`로
  넣어서, 모델이 "이 달 값은 리뷰 1개짜리라 신뢰도가 낮다"를 학습에 반영하게 한다 (조님 06b 3차 시도 방식)
- 각 상품 시리즈의 앞 75%로 한 번만 학습(재학습 없음, `retrain=False`), 나머지 뒤 25% 구간을
  `stride=OUTPUT_LEN`씩 밀어가며 반복 예측 -> 실제값과 비교 (`darts.historical_forecasts`)
- **naive(직전 관측값)를 항상 같이 계산해서 비교 기준으로 삼는다**

In [4]:
from darts.models import DLinearModel, NHiTSModel
from darts.utils.likelihood_models import QuantileRegression

STRIDE = OUTPUT_LEN

dlinear = DLinearModel(input_chunk_length=INPUT_LEN, output_chunk_length=OUTPUT_LEN,
                        n_epochs=50, random_state=42,
                        pl_trainer_kwargs={"enable_progress_bar": False, "enable_model_summary": False, "accelerator": "cpu"})
dlinear.fit(series=train_target)
dlinear_hist = dlinear.historical_forecasts(
    series=series_scaled, forecast_horizon=OUTPUT_LEN, stride=STRIDE, start=TRAIN_FRAC,
    retrain=False, last_points_only=False, verbose=False,
)

nhits = NHiTSModel(input_chunk_length=INPUT_LEN, output_chunk_length=OUTPUT_LEN,
                    n_epochs=75, random_state=42,
                    likelihood=QuantileRegression(quantiles=[0.1, 0.5, 0.9]),
                    pl_trainer_kwargs={"enable_progress_bar": False, "enable_model_summary": False, "accelerator": "cpu"})
nhits.fit(series=train_target, past_covariates=cov_scaled)
nhits_hist = nhits.historical_forecasts(
    series=series_scaled, past_covariates=cov_scaled,
    forecast_horizon=OUTPUT_LEN, stride=STRIDE, start=TRAIN_FRAC,
    retrain=False, last_points_only=False, verbose=False, num_samples=200,
)
print("백테스트 완료")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/utilities/data.py:106: Total length of `list` across ranks is zero. Please make sure this was your intention.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

`Trainer.fit` stopped: `max_epochs=75` reached.


GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


백테스트 완료


## 4. 롤링 윈도우별 비교: naive 포함

In [5]:
dlinear_errs, nhits_errs, naive_errs = [], [], []
n_windows = 0
for i, (dl_windows, nh_windows) in enumerate(zip(dlinear_hist, nhits_hist)):
    full_pd = series_list[i].to_series()
    for dl_w, nh_w in zip(dl_windows, nh_windows):
        actual = full_pd.reindex(dl_w.time_index).values

        dl_pred = dl_w.values().ravel() * (train_maxs[i] - train_mins[i]) + train_mins[i]
        nh_pred = nh_w.quantile(0.5).values().ravel() * (train_maxs[i] - train_mins[i]) + train_mins[i]

        naive_origin = dl_w.time_index[0] - pd.DateOffset(months=1)
        naive_val = full_pd.get(naive_origin, np.nan)
        if pd.isna(naive_val):
            continue

        n_windows += 1
        dlinear_errs.append(np.abs(actual - dl_pred).mean())
        nhits_errs.append(np.abs(actual - nh_pred).mean())
        naive_errs.append(np.abs(actual - naive_val).mean())

dlinear_errs, nhits_errs, naive_errs = map(np.array, (dlinear_errs, nhits_errs, naive_errs))
print(f"총 롤링 윈도우 {n_windows}개 (상품 {len(names)}개)")
print(f"naive 평균 MAE:   {naive_errs.mean():.4f}")
print(f"DLinear 평균 MAE: {dlinear_errs.mean():.4f}")
print(f"NHiTS 평균 MAE:   {nhits_errs.mean():.4f}")
print()
print(f"DLinear가 naive보다 나은 윈도우: {(dlinear_errs < naive_errs).mean()*100:.1f}%")
print(f"NHiTS가 naive보다 나은 윈도우:   {(nhits_errs < naive_errs).mean()*100:.1f}%")

총 롤링 윈도우 234개 (상품 85개)
naive 평균 MAE:   0.1456
DLinear 평균 MAE: 0.1556
NHiTS 평균 MAE:   0.1493

DLinear가 naive보다 나은 윈도우: 37.6%
NHiTS가 naive보다 나은 윈도우:   42.7%


## 5. 조기탐지 후보: 노이즈 필터 + "상품 자체 역대 최고치" 비교

검증 기간(뒤 25%) 실제값이 NHiTS 90% 예측구간 상한을 넘은 경우를 1차 후보로 뽑는다. 다만 리뷰 수가
적은 달은 표본 노이즈로 우연히 튈 수 있어(조님 06b/06c에서도 동일 문제), **① 그 시점 실제 리뷰가
4건 이상인 것만 신뢰 가능한 후보로 필터링**하고, **② "모델이 못 맞췄다"와 "진짜 급증"은 다른 질문이라**
그 상품 자체의 (검증구간을 뺀) 과거 평균/표준편차·역대 최고치보다도 실제값이 높은지까지 확인한다.

In [6]:
MIN_ALERT_REVIEWS = 4

alerts, alerts_noisy = [], []
for i, (name, nh_windows) in enumerate(zip(names, nhits_hist)):
    full_pd = series_list[i].to_series()
    full_counts = count_series_list[i].to_series()
    cut = int(len(series_list[i]) * TRAIN_FRAC)
    hist_vals = series_list[i].values().ravel()[:cut]  # 검증구간 제외한 그 상품 자체 과거 값
    own_max = hist_vals.max()
    own_mean, own_std = hist_vals.mean(), hist_vals.std()

    for nh_w in nh_windows:
        lo = nh_w.quantile(0.1).values().ravel()
        hi = nh_w.quantile(0.9).values().ravel()
        actual = full_pd.reindex(nh_w.time_index).values
        rev_counts = full_counts.reindex(nh_w.time_index).values

        for t, a, h, rc in zip(nh_w.time_index, actual, hi, rev_counts):
            h_real = h * (train_maxs[i] - train_mins[i]) + train_mins[i]
            if a <= h_real:
                continue
            row = {
                "product": name, "month": t, "actual": a, "upper_bound_real": h_real,
                "review_count": int(rc),
                "상품자체_역대최고치_초과": a > own_max,
                "상품자체_평균+1.5표준편차_초과": a > own_mean + 1.5 * own_std,
            }
            (alerts if rc >= MIN_ALERT_REVIEWS else alerts_noisy).append(row)

alerts_df = pd.DataFrame(alerts)
alerts_noisy_df = pd.DataFrame(alerts_noisy)
print(f"1차 후보(90% 예측구간 상한 초과): 신뢰 가능(리뷰 {MIN_ALERT_REVIEWS}건 이상) {len(alerts_df)}건, "
      f"노이즈로 제외(리뷰 {MIN_ALERT_REVIEWS}건 미만) {len(alerts_noisy_df)}건")
if len(alerts_df):
    n_new_high = alerts_df["상품자체_역대최고치_초과"].sum()
    print(f"그 중 '상품 자체 역대 최고치'보다도 높은 진짜 신규 급증: {n_new_high}건 ({n_new_high/len(alerts_df)*100:.0f}%)")

1차 후보(90% 예측구간 상한 초과): 신뢰 가능(리뷰 4건 이상) 55건, 노이즈로 제외(리뷰 4건 미만) 83건
그 중 '상품 자체 역대 최고치'보다도 높은 진짜 신규 급증: 7건 (13%)


## 6. 결과 요약

In [7]:
summary = pd.DataFrame([{
    "대상 상품 수": len(names),
    "롤링 윈도우 수": n_windows,
    "naive_MAE": naive_errs.mean(),
    "DLinear_MAE": dlinear_errs.mean(),
    "NHiTS_MAE": nhits_errs.mean(),
    "DLinear_naive대비승률(%)": (dlinear_errs < naive_errs).mean() * 100,
    "NHiTS_naive대비승률(%)": (nhits_errs < naive_errs).mean() * 100,
    "조기탐지_신뢰가능": len(alerts_df),
    "조기탐지_노이즈제외": len(alerts_noisy_df),
    "조기탐지_상품자체역대최고치초과": int(alerts_df["상품자체_역대최고치_초과"].sum()) if len(alerts_df) else 0,
}])
summary

,대상 상품 수,롤링 윈도우 수,naive_MAE,DLinear_MAE,NHiTS_MAE,DLinear_naive대비승률(%),NHiTS_naive대비승률(%),조기탐지_신뢰가능,조기탐지_노이즈제외,조기탐지_상품자체역대최고치초과
0,85,234,0.145585,0.155607,0.149272,37.606838,42.735043,55,83,7


## 7. 피처 보강: 카테고리(static covariate) + 월 계절성(future covariate) 추가

지금까지 NHiTS는 리뷰수(past_covariate)만 추가 정보로 받았다. 새 모델 대신, 기존 NHiTS에 정보를 더 줘서
성능이 실제로 개선되는지 검증한다.

- **카테고리 static covariate**: 옷/신발/잡화 3그룹 원핫. 지금까지는 85개 상품을 다 섞어서 학습시켜서
  모델이 "이게 옷인지 신발인지"를 모른 채 배웠다. 5단계에서 카테고리별 어휘·불만 유형이 확연히 다르다는
  걸 이미 확인했으므로(`05_토픽모델링_손.md`), 이 정보를 명시적으로 주면 도움이 될 수 있다.
- **월 계절성 future covariate**: `datetime_attribute_timeseries(..., cyclic=True)`로 월을 sin/cos
  인코딩. 계절(겨울 패딩, 여름 원피스 등)에 따라 결함 유형·비율이 달라질 수 있어서 추가.

In [8]:
# 상품별 카테고리(MainCategory) 조회 -- 옷(여성의류+남성의류)/신발(패션슈즈)/잡화, 5단계와 동일 매핑
cat_map_raw = sub.groupby("ProductName")["MainCategory"].agg(lambda s: s.mode().iloc[0])

def to_group(mc):
    if mc in ("여성의류", "남성의류"):
        return "옷"
    if mc == "패션슈즈":
        return "신발"
    return "잡화"

cat_map = cat_map_raw.map(to_group)

from darts.utils.timeseries_generation import datetime_attribute_timeseries

# NHiTS는 future_covariates를 지원하지 않으므로, 월 sin/cos도 past_covariate로 합쳐서 넣는다
# (그 달 리뷰수 covariate와 같은 시간축이라 stack으로 붙일 수 있음)
series_scaled_feat, train_target_feat, cov_scaled_feat = [], [], []
for name, s_scaled, t_target, cov in zip(names, series_scaled, train_target, cov_scaled):
    group = cat_map[name]
    static_df = pd.DataFrame({
        "is_옷": [1.0 if group == "옷" else 0.0],
        "is_신발": [1.0 if group == "신발" else 0.0],
        "is_잡화": [1.0 if group == "잡화" else 0.0],
    })
    series_scaled_feat.append(s_scaled.with_static_covariates(static_df))
    train_target_feat.append(t_target.with_static_covariates(static_df))

    month_cov = datetime_attribute_timeseries(cov.time_index, attribute="month", cyclic=True)
    cov_scaled_feat.append(cov.stack(month_cov))

print(f"대상 {len(names)}개 상품 카테고리 분포: {cat_map.reindex(names).value_counts().to_dict()}")
print("카테고리 static covariate + 월 sin/cos(past covariate로 합침) 준비 완료")
print("past covariate 차원:", cov_scaled_feat[0].n_components, "(리뷰수 1 + month_sin/cos 2)")

대상 85개 상품 카테고리 분포: {'옷': 57, '신발': 21, '잡화': 7}
카테고리 static covariate + 월 sin/cos(past covariate로 합침) 준비 완료
past covariate 차원: 3 (리뷰수 1 + month_sin/cos 2)


In [9]:
nhits2 = NHiTSModel(input_chunk_length=INPUT_LEN, output_chunk_length=OUTPUT_LEN,
                     n_epochs=75, random_state=42,
                     likelihood=QuantileRegression(quantiles=[0.1, 0.5, 0.9]),
                     pl_trainer_kwargs={"enable_progress_bar": False, "enable_model_summary": False, "accelerator": "cpu"})
nhits2.fit(series=train_target_feat, past_covariates=cov_scaled_feat)
nhits2_hist = nhits2.historical_forecasts(
    series=series_scaled_feat, past_covariates=cov_scaled_feat,
    forecast_horizon=OUTPUT_LEN, stride=STRIDE, start=TRAIN_FRAC,
    retrain=False, last_points_only=False, verbose=False, num_samples=200,
)
print("피처 보강 NHiTS(v2) 백테스트 완료")

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/utilities/data.py:106: Total length of `list` across ranks is zero. Please make sure this was your intention.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


`Trainer.fit` stopped: `max_epochs=75` reached.


GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


피처 보강 NHiTS(v2) 백테스트 완료


In [10]:
nhits2_errs, naive_errs2, n_windows2 = [], [], 0
for i, nh_windows in enumerate(nhits2_hist):
    full_pd = series_list[i].to_series()
    for nh_w in nh_windows:
        actual = full_pd.reindex(nh_w.time_index).values
        nh_pred = nh_w.quantile(0.5).values().ravel() * (train_maxs[i] - train_mins[i]) + train_mins[i]

        naive_origin = nh_w.time_index[0] - pd.DateOffset(months=1)
        naive_val = full_pd.get(naive_origin, np.nan)
        if pd.isna(naive_val):
            continue

        n_windows2 += 1
        nhits2_errs.append(np.abs(actual - nh_pred).mean())
        naive_errs2.append(np.abs(actual - naive_val).mean())

nhits2_errs, naive_errs2 = np.array(nhits2_errs), np.array(naive_errs2)
print(f"윈도우 {n_windows2}개 (기존 234개와 동일해야 정상)")
print()
print(f"naive 평균 MAE:            {naive_errs2.mean():.4f}")
print(f"NHiTS(원본, 리뷰수만) MAE:  {nhits_errs.mean():.4f}")
print(f"NHiTS(피처보강) MAE:        {nhits2_errs.mean():.4f}")
print()
print(f"피처보강이 원본 NHiTS보다 나은 윈도우: {(nhits2_errs < nhits_errs).mean()*100:.1f}%")
print(f"피처보강이 naive보다 나은 윈도우:      {(nhits2_errs < naive_errs2).mean()*100:.1f}%")

윈도우 234개 (기존 234개와 동일해야 정상)

naive 평균 MAE:            0.1456
NHiTS(원본, 리뷰수만) MAE:  0.1493
NHiTS(피처보강) MAE:        0.1498

피처보강이 원본 NHiTS보다 나은 윈도우: 41.5%
피처보강이 naive보다 나은 윈도우:      43.6%


## 8. 데이터 확장 1차: 기준 완화 (패션 내에서만, 리뷰 30건→20건)

지금까지(85개 상품)는 naive를 못 이겼는데, 원인이 "모델/피처 문제"가 아니라 "학습 데이터 자체가 부족해서"일
수 있다. 같은 패션 도메인 안에서 리뷰 최소 기준만 30건→20건으로 낮춰서 상품 수를 늘리고, 나머지 파이프라인
(leakage-safe 스케일링, retrain=False 롤링 백테스트, naive 비교)은 그대로 유지해서 데이터 양 자체의 효과만
비교한다.

In [11]:
MIN_REVIEWS_20 = 20

counts_by_product_20 = fashion.groupby("ProductName").size()
qualifying_20 = counts_by_product_20[counts_by_product_20 >= MIN_REVIEWS_20].index
print(f"리뷰 {MIN_REVIEWS_20}건 이상 상품: {len(qualifying_20)}개 (기존 30건 기준 {len(qualifying)}개)")

sub20 = fashion[fashion["ProductName"].isin(qualifying_20)].copy()

def build_product_series_generic(sub_df, min_reviews_label, min_months):
    names_, series_list_, count_series_list_ = [], [], []
    skipped_short = 0
    for name, g in sub_df.groupby("ProductName"):
        m = g.groupby(g["RDate_parsed"].dt.to_period("M").dt.to_timestamp()).agg(
            total=("review_id", "count"),
            negative=("GeneralPolarity", lambda s: (s == -1).sum()),
        )
        m["neg_ratio"] = m["negative"] / m["total"]
        full_idx = pd.date_range(m.index.min(), m.index.max(), freq="MS")
        if len(full_idx) < min_months:
            skipped_short += 1
            continue
        ratio = m["neg_ratio"].reindex(full_idx).interpolate(limit_direction="both")
        counts = m["total"].reindex(full_idx).fillna(0)
        names_.append(name)
        series_list_.append(TimeSeries.from_times_and_values(full_idx, ratio.values))
        count_series_list_.append(TimeSeries.from_times_and_values(full_idx, counts.values))
    print(f"관측기간 {min_months}개월 이상: {len(series_list_)}개 사용 (짧아서 제외 {skipped_short}개)")
    return names_, series_list_, count_series_list_

names20, series_list20, count_series_list20 = build_product_series_generic(sub20, MIN_REVIEWS_20, MIN_MONTHS)

리뷰 20건 이상 상품: 525개 (기존 30건 기준 332개)


관측기간 10개월 이상: 129개 사용 (짧아서 제외 396개)


In [12]:
# 동일한 leakage-safe 스케일링 (학습구간 min/max만 사용) + 학습 가능 길이 필터
train_mins20, train_maxs20, series_scaled20, cov_scaled20, train_target20 = [], [], [], [], []
kept_names20, kept_series_list20, kept_count_series_list20 = [], [], []
skipped_train_too_short20 = 0
for name, s, c in zip(names20, series_list20, count_series_list20):
    cut = int(len(s) * TRAIN_FRAC)
    if cut < MIN_TRAIN_LEN or (len(s) - cut) < OUTPUT_LEN:
        skipped_train_too_short20 += 1
        continue
    tmin, tmax = float(s[:cut].values().min()), float(s[:cut].values().max())
    cmin, cmax = float(c[:cut].values().min()), float(c[:cut].values().max())
    train_mins20.append(tmin); train_maxs20.append(tmax)
    series_scaled20.append(manual_scale(s, tmin, tmax))
    cov_scaled20.append(manual_scale(c, cmin, cmax))
    train_target20.append(manual_scale(s, tmin, tmax)[:cut])
    kept_names20.append(name)
    kept_series_list20.append(s)
    kept_count_series_list20.append(c)

names20, series_list20, count_series_list20 = kept_names20, kept_series_list20, kept_count_series_list20
print(f"학습구간 부족으로 추가 제외: {skipped_train_too_short20}개")
print(f"최종 사용 상품(리뷰20건+): {len(series_scaled20)}개 (기존 리뷰30건+ 기준 {len(series_scaled)}개)")

학습구간 부족으로 추가 제외: 19개
최종 사용 상품(리뷰20건+): 110개 (기존 리뷰30건+ 기준 85개)


In [13]:
dlinear20 = DLinearModel(input_chunk_length=INPUT_LEN, output_chunk_length=OUTPUT_LEN,
                          n_epochs=50, random_state=42,
                          pl_trainer_kwargs={"enable_progress_bar": False, "enable_model_summary": False, "accelerator": "cpu"})
dlinear20.fit(series=train_target20)
dlinear20_hist = dlinear20.historical_forecasts(
    series=series_scaled20, forecast_horizon=OUTPUT_LEN, stride=STRIDE, start=TRAIN_FRAC,
    retrain=False, last_points_only=False, verbose=False,
)

nhits20 = NHiTSModel(input_chunk_length=INPUT_LEN, output_chunk_length=OUTPUT_LEN,
                      n_epochs=75, random_state=42,
                      likelihood=QuantileRegression(quantiles=[0.1, 0.5, 0.9]),
                      pl_trainer_kwargs={"enable_progress_bar": False, "enable_model_summary": False, "accelerator": "cpu"})
nhits20.fit(series=train_target20, past_covariates=cov_scaled20)
nhits20_hist = nhits20.historical_forecasts(
    series=series_scaled20, past_covariates=cov_scaled20,
    forecast_horizon=OUTPUT_LEN, stride=STRIDE, start=TRAIN_FRAC,
    retrain=False, last_points_only=False, verbose=False, num_samples=200,
)
print("리뷰20건+ 백테스트 완료")

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/utilities/data.py:106: Total length of `list` across ranks is zero. Please make sure this was your intention.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


`Trainer.fit` stopped: `max_epochs=50` reached.


...

`Trainer.fit` stopped: `max_epochs=75` reached.


GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


리뷰20건+ 백테스트 완료


In [14]:
dlinear20_errs, nhits20_errs, naive20_errs = [], [], []
n_windows20 = 0
for i, (dl_windows, nh_windows) in enumerate(zip(dlinear20_hist, nhits20_hist)):
    full_pd = series_list20[i].to_series()
    for dl_w, nh_w in zip(dl_windows, nh_windows):
        actual = full_pd.reindex(dl_w.time_index).values
        dl_pred = dl_w.values().ravel() * (train_maxs20[i] - train_mins20[i]) + train_mins20[i]
        nh_pred = nh_w.quantile(0.5).values().ravel() * (train_maxs20[i] - train_mins20[i]) + train_mins20[i]

        naive_origin = dl_w.time_index[0] - pd.DateOffset(months=1)
        naive_val = full_pd.get(naive_origin, np.nan)
        if pd.isna(naive_val):
            continue

        n_windows20 += 1
        dlinear20_errs.append(np.abs(actual - dl_pred).mean())
        nhits20_errs.append(np.abs(actual - nh_pred).mean())
        naive20_errs.append(np.abs(actual - naive_val).mean())

dlinear20_errs, nhits20_errs, naive20_errs = map(np.array, (dlinear20_errs, nhits20_errs, naive20_errs))
print(f"총 롤링 윈도우 {n_windows20}개 (상품 {len(names20)}개, 기존 {n_windows}개/{len(names)}개 대비)")
print()
print(f"naive 평균 MAE:   {naive20_errs.mean():.4f}  (기존 리뷰30건+: {naive_errs.mean():.4f})")
print(f"DLinear 평균 MAE: {dlinear20_errs.mean():.4f}  (기존: {dlinear_errs.mean():.4f})")
print(f"NHiTS 평균 MAE:   {nhits20_errs.mean():.4f}  (기존: {nhits_errs.mean():.4f})")
print()
print(f"DLinear가 naive보다 나은 윈도우: {(dlinear20_errs < naive20_errs).mean()*100:.1f}% (기존 {(dlinear_errs < naive_errs).mean()*100:.1f}%)")
print(f"NHiTS가 naive보다 나은 윈도우:   {(nhits20_errs < naive20_errs).mean()*100:.1f}% (기존 {(nhits_errs < naive_errs).mean()*100:.1f}%)")

총 롤링 윈도우 293개 (상품 110개, 기존 234개/85개 대비)

naive 평균 MAE:   0.1406  (기존 리뷰30건+: 0.1456)
DLinear 평균 MAE: 0.1522  (기존: 0.1556)
NHiTS 평균 MAE:   0.1404  (기존: 0.1493)

DLinear가 naive보다 나은 윈도우: 36.9% (기존 37.6%)
NHiTS가 naive보다 나은 윈도우:   39.9% (기존 42.7%)


## 9. 데이터 확장 2차: 5개 도메인 전체 통합 전역학습

Ch.19 전역학습의 취지는 "여러 series를 같이 학습시켜 패턴을 공유"하는 것이라, 패션에만 한정할 이유가 없다.
5개 도메인(패션/생활/화장품/IT기기/가전) 상품(리뷰 20건+, 관측기간 10개월+)을 전부 한 모델에 넣어 같이
학습시키고(도메인을 static covariate로 명시), **평가는 패션 상품에 대해서만** 한다 -- 최종 산출물은 패션
결함 조기탐지이므로, 다른 도메인 데이터는 어디까지나 "같이 학습시켜 패턴을 전이받는" 보조 재료다.

In [15]:
ALL_DOMAINS = ["패션", "생활", "화장품", "IT기기", "가전"]
all_reviews = reviews[reviews["RDate_parsed"].notna()].copy()
# ProductName이 도메인 간 극소수(16건) 겹치는 경우가 있어 (도메인, 상품명) 조합을 고유 키로 사용
all_reviews["product_key"] = all_reviews["Domain"] + "||" + all_reviews["ProductName"]

counts_all = all_reviews.groupby("product_key").size()
qualifying_all = counts_all[counts_all >= MIN_REVIEWS_20].index
sub_all = all_reviews[all_reviews["product_key"].isin(qualifying_all)].copy()
print(f"5개 도메인 합계, 리뷰 {MIN_REVIEWS_20}건 이상 상품: {len(qualifying_all)}개")

names_all, series_all, count_series_all, domain_of = [], [], [], {}
skipped_short_all = 0
for key, g in sub_all.groupby("product_key"):
    m = g.groupby(g["RDate_parsed"].dt.to_period("M").dt.to_timestamp()).agg(
        total=("review_id", "count"),
        negative=("GeneralPolarity", lambda s: (s == -1).sum()),
    )
    m["neg_ratio"] = m["negative"] / m["total"]
    full_idx = pd.date_range(m.index.min(), m.index.max(), freq="MS")
    if len(full_idx) < MIN_MONTHS:
        skipped_short_all += 1
        continue
    ratio = m["neg_ratio"].reindex(full_idx).interpolate(limit_direction="both")
    counts = m["total"].reindex(full_idx).fillna(0)
    names_all.append(key)
    series_all.append(TimeSeries.from_times_and_values(full_idx, ratio.values))
    count_series_all.append(TimeSeries.from_times_and_values(full_idx, counts.values))
    domain_of[key] = g["Domain"].iloc[0]

print(f"관측기간 {MIN_MONTHS}개월 이상: {len(series_all)}개 사용 (짧아서 제외 {skipped_short_all}개)")
from collections import Counter
print("도메인별 분포:", Counter(domain_of[n] for n in names_all))

5개 도메인 합계, 리뷰 20건 이상 상품: 2037개


관측기간 10개월 이상: 1236개 사용 (짧아서 제외 801개)
도메인별 분포: Counter({'생활': 337, '화장품': 278, '가전': 260, 'IT기기': 232, '패션': 129})


In [16]:
# leakage-safe 스케일링 + 학습가능 길이 필터 (기존과 동일 로직) + 도메인 static covariate 부착
train_mins_all, train_maxs_all, series_scaled_all, cov_scaled_all, train_target_all = [], [], [], [], []
kept_names_all, kept_series_all, kept_domain_all = [], [], []
skipped_train_all = 0
for name, s, c in zip(names_all, series_all, count_series_all):
    cut = int(len(s) * TRAIN_FRAC)
    if cut < MIN_TRAIN_LEN or (len(s) - cut) < OUTPUT_LEN:
        skipped_train_all += 1
        continue
    tmin, tmax = float(s[:cut].values().min()), float(s[:cut].values().max())
    cmin, cmax = float(c[:cut].values().min()), float(c[:cut].values().max())

    dom = domain_of[name]
    static_df = pd.DataFrame({f"dom_{d}": [1.0 if d == dom else 0.0] for d in ALL_DOMAINS})

    s_scaled = manual_scale(s, tmin, tmax).with_static_covariates(static_df)
    t_target = manual_scale(s, tmin, tmax)[:cut].with_static_covariates(static_df)
    c_scaled = manual_scale(c, cmin, cmax)

    train_mins_all.append(tmin); train_maxs_all.append(tmax)
    series_scaled_all.append(s_scaled); cov_scaled_all.append(c_scaled); train_target_all.append(t_target)
    kept_names_all.append(name); kept_series_all.append(s); kept_domain_all.append(dom)

names_all, series_all = kept_names_all, kept_series_all
print(f"학습구간 부족으로 추가 제외: {skipped_train_all}개")
print(f"최종 전역학습 대상(5개 도메인 합계): {len(series_scaled_all)}개")
print("도메인별:", Counter(kept_domain_all))

학습구간 부족으로 추가 제외: 80개
최종 전역학습 대상(5개 도메인 합계): 1156개
도메인별: Counter({'생활': 324, '화장품': 271, '가전': 232, 'IT기기': 219, '패션': 110})


In [17]:
nhits_global = NHiTSModel(input_chunk_length=INPUT_LEN, output_chunk_length=OUTPUT_LEN,
                           n_epochs=75, random_state=42,
                           likelihood=QuantileRegression(quantiles=[0.1, 0.5, 0.9]),
                           pl_trainer_kwargs={"enable_progress_bar": False, "enable_model_summary": False, "accelerator": "cpu"})
nhits_global.fit(series=train_target_all, past_covariates=cov_scaled_all)
print("전역학습(5개 도메인 통합) 완료 -- 대상", len(train_target_all), "개 시리즈")

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/utilities/data.py:106: Total length of `list` across ranks is zero. Please make sure this was your intention.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


`Trainer.fit` stopped: `max_epochs=75` reached.


전역학습(5개 도메인 통합) 완료 -- 대상 1156 개 시리즈


In [18]:
# 평가는 패션 상품에 대해서만: 전역학습된 모델에 패션 시리즈만 넣어 백테스트
fashion_idx_in_all = [i for i, d in enumerate(kept_domain_all) if d == "패션"]
fashion_series_scaled_all = [series_scaled_all[i] for i in fashion_idx_in_all]
fashion_cov_scaled_all = [cov_scaled_all[i] for i in fashion_idx_in_all]
fashion_series_all = [series_all[i] for i in fashion_idx_in_all]
fashion_train_mins_all = [train_mins_all[i] for i in fashion_idx_in_all]
fashion_train_maxs_all = [train_maxs_all[i] for i in fashion_idx_in_all]
print(f"평가 대상 패션 상품: {len(fashion_idx_in_all)}개")

nhits_global_hist = nhits_global.historical_forecasts(
    series=fashion_series_scaled_all, past_covariates=fashion_cov_scaled_all,
    forecast_horizon=OUTPUT_LEN, stride=STRIDE, start=TRAIN_FRAC,
    retrain=False, last_points_only=False, verbose=False, num_samples=200,
)
print("패션 상품 백테스트(전역학습 모델 기준) 완료")

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


평가 대상 패션 상품: 110개


패션 상품 백테스트(전역학습 모델 기준) 완료


In [19]:
nhits_global_errs, naive_global_errs, n_windows_global = [], [], 0
for i, nh_windows in enumerate(nhits_global_hist):
    full_pd = fashion_series_all[i].to_series()
    for nh_w in nh_windows:
        actual = full_pd.reindex(nh_w.time_index).values
        nh_pred = nh_w.quantile(0.5).values().ravel() * (fashion_train_maxs_all[i] - fashion_train_mins_all[i]) + fashion_train_mins_all[i]

        naive_origin = nh_w.time_index[0] - pd.DateOffset(months=1)
        naive_val = full_pd.get(naive_origin, np.nan)
        if pd.isna(naive_val):
            continue

        n_windows_global += 1
        nhits_global_errs.append(np.abs(actual - nh_pred).mean())
        naive_global_errs.append(np.abs(actual - naive_val).mean())

nhits_global_errs, naive_global_errs = map(np.array, (nhits_global_errs, naive_global_errs))
print(f"패션 평가 윈도우 {n_windows_global}개 (상품 {len(fashion_idx_in_all)}개)")
print()
print(f"naive 평균 MAE:              {naive_global_errs.mean():.4f}")
print(f"NHiTS(5도메인 전역학습) MAE:  {nhits_global_errs.mean():.4f}")
print()
print(f"전역학습 NHiTS가 naive보다 나은 윈도우: {(nhits_global_errs < naive_global_errs).mean()*100:.1f}%")
print()
print("--- 지금까지 전체 비교 ---")
print(f"naive (리뷰30+, 85개):            {naive_errs.mean():.4f}")
print(f"NHiTS (리뷰30+, 85개, 패션전용):    {nhits_errs.mean():.4f}")
print(f"NHiTS (리뷰20+, 110개, 패션전용):   {nhits20_errs.mean():.4f}")
print(f"NHiTS (리뷰20+, 5도메인 전역학습):  {nhits_global_errs.mean():.4f}")

패션 평가 윈도우 293개 (상품 110개)

naive 평균 MAE:              0.1406
NHiTS(5도메인 전역학습) MAE:  0.1330

전역학습 NHiTS가 naive보다 나은 윈도우: 43.0%

--- 지금까지 전체 비교 ---
naive (리뷰30+, 85개):            0.1456
NHiTS (리뷰30+, 85개, 패션전용):    0.1493
NHiTS (리뷰20+, 110개, 패션전용):   0.1404
NHiTS (리뷰20+, 5도메인 전역학습):  0.1330


## 10. 조기탐지 후보 재계산 (전역학습 모델 기준)

섹션 5의 조기탐지 후보는 naive에게 진 1차 모델(패션 85개 단독 학습) 기준이었다. naive를 실제로 이긴
섹션 9의 5도메인 전역학습 모델(`nhits_global`)로 다시 뽑는다. 로직은 섹션 5와 동일: ① 90% 예측구간
상한 초과, ② 그 시점 리뷰 4건 이상(노이즈 필터), ③ 상품 자체 (검증구간 제외) 과거 평균/최고치보다도
높은지까지 확인.

In [20]:
# 섹션9에서 패션 상품의 리뷰수(raw, 비스케일) 시리즈를 따로 안 남겨뒀어서 다시 구성
fashion_names_all = [kept_names_all[i] for i in fashion_idx_in_all]
fashion_count_series_all = []
for key in fashion_names_all:
    g = sub_all[sub_all["product_key"] == key]
    m = g.groupby(g["RDate_parsed"].dt.to_period("M").dt.to_timestamp()).size()
    idx_full = pd.date_range(m.index.min(), m.index.max(), freq="MS")
    counts = m.reindex(idx_full, fill_value=0)
    fashion_count_series_all.append(TimeSeries.from_times_and_values(idx_full, counts.values))

print(f"패션 리뷰수 시리즈 재구성 완료: {len(fashion_count_series_all)}개")

패션 리뷰수 시리즈 재구성 완료: 110개


In [21]:
MIN_ALERT_REVIEWS = 4

alerts_v3, alerts_v3_noisy = [], []
for i, (name, nh_windows) in enumerate(zip(fashion_names_all, nhits_global_hist)):
    full_pd = fashion_series_all[i].to_series()
    full_counts = fashion_count_series_all[i].to_series()
    cut = int(len(fashion_series_all[i]) * TRAIN_FRAC)
    hist_vals = fashion_series_all[i].values().ravel()[:cut]
    own_max = hist_vals.max()
    own_mean, own_std = hist_vals.mean(), hist_vals.std()

    for nh_w in nh_windows:
        hi = nh_w.quantile(0.9).values().ravel()
        actual = full_pd.reindex(nh_w.time_index).values
        rev_counts = full_counts.reindex(nh_w.time_index).values

        for t, a, h, rc in zip(nh_w.time_index, actual, hi, rev_counts):
            h_real = h * (fashion_train_maxs_all[i] - fashion_train_mins_all[i]) + fashion_train_mins_all[i]
            if a <= h_real:
                continue
            row = {
                "product": name.replace("패션||", ""), "month": t, "actual": a, "upper_bound_real": h_real,
                "review_count": int(rc),
                "상품자체_역대최고치_초과": a > own_max,
                "상품자체_평균+1.5표준편차_초과": a > own_mean + 1.5 * own_std,
            }
            (alerts_v3 if rc >= MIN_ALERT_REVIEWS else alerts_v3_noisy).append(row)

alerts_v3_df = pd.DataFrame(alerts_v3)
alerts_v3_noisy_df = pd.DataFrame(alerts_v3_noisy)
print(f"[전역학습 모델] 1차 후보(90% 예측구간 상한 초과): 신뢰 가능(리뷰 {MIN_ALERT_REVIEWS}건 이상) {len(alerts_v3_df)}건, "
      f"노이즈로 제외 {len(alerts_v3_noisy_df)}건")
if len(alerts_v3_df):
    n_new_high = alerts_v3_df["상품자체_역대최고치_초과"].sum()
    print(f"그 중 '상품 자체 역대 최고치'보다도 높은 진짜 신규 급증: {n_new_high}건 ({n_new_high/len(alerts_v3_df)*100:.0f}%)")
print()
print(f"(참고, 섹션5 1차 모델 기준: 신뢰가능 55건, 진짜신규급증 7건/13%)")

[전역학습 모델] 1차 후보(90% 예측구간 상한 초과): 신뢰 가능(리뷰 4건 이상) 53건, 노이즈로 제외 75건
그 중 '상품 자체 역대 최고치'보다도 높은 진짜 신규 급증: 10건 (19%)

(참고, 섹션5 1차 모델 기준: 신뢰가능 55건, 진짜신규급증 7건/13%)


In [22]:
alerts_v3_df.sort_values("actual", ascending=False).head(10)

,product,month,actual,upper_bound_real,review_count,상품자체_역대최고치_초과,상품자체_평균+1.5표준편차_초과
22,OO 볼** 소가죽 여성 캐주얼부츠,2019-12-01,1.000000,0.001034,5,True,True
18,OO 보** 여성샌들 + 슬리퍼 2종,2021-07-01,0.800000,0.284383,5,True,True
0,OO 18K 총 6종 18K 빅** 목걸이 세트,2021-02-01,0.777778,0.422660,9,False,False
26,OO 소가죽 펌프스 샌들,2021-10-01,0.750000,0.309259,4,False,True
43,OO 여성 썸머 시** 팬츠 3종,2022-02-01,0.600000,0.397496,5,False,False
30,OO 여름 에** 재킷 2종,2021-06-01,0.571429,0.430108,7,False,True
33,OO 여성 데님팬츠 3종,2022-02-01,0.555556,0.153182,9,False,False
35,OO 여성 매** 스트레치 팬츠 4종,2020-05-01,0.512821,0.024262,39,False,False
4,OO 남성 봄 풀오버 니트 3종,2021-09-01,0.500000,0.007783,4,False,True
17,OO 보** 여성샌들 + 슬리퍼 2종,2021-06-01,0.500000,0.184256,4,True,True


## 11. 도메인통합 모델 + 피처보강 재시도 (B)

섹션7의 피처보강(카테고리+월계절성)은 패션 단독(85개, 중앙값 15개월)에서는 효과가 없었다 -- 계절 주기가
한 바퀴도 안 돌 만큼 데이터가 부족했을 가능성이 있었다. 지금은 5도메인 통합으로 1,156개 시리즈까지
늘었으니(섹션9), 같은 아이디어(카테고리 static covariate + 월 sin/cos past covariate)를 이 위에 다시
얹어서 효과가 달라지는지 확인한다. **비교 기준은 섹션9의 `nhits_global`(naive 대비 5.4% 개선)이다.**

In [23]:
# 전체 도메인 상품의 세부 카테고리(MainCategory, 20종) -- 도메인(5) + 세부카테고리(20) 둘 다 static covariate로
cat_map_all = sub_all.groupby("product_key")["MainCategory"].agg(lambda s: s.mode().iloc[0])
ALL_CATEGORIES = sorted(sub_all["MainCategory"].unique())
print(f"세부 카테고리 {len(ALL_CATEGORIES)}종: {ALL_CATEGORIES}")

세부 카테고리 20종: ['계절가전', '남성의류', '남성화장품', '메이크업/뷰티소품', '생활/미용/욕실가전', '세제/세정/탈취제', '스킨케어', '여성의류', '영상/음향가전', '위생용품', '자동차기기', '잡화', '주방가전', '주방용품', '청소/세탁용품', '카메라/게임기/태블릿', '컴퓨터/주변기기', '패션슈즈', '헤어/바디케어', '휴대폰/주변기기']


In [24]:
series_scaled_all_feat, train_target_all_feat, cov_scaled_all_feat = [], [], []
for name, s_scaled, t_target, cov, dom in zip(names_all, series_scaled_all, train_target_all, cov_scaled_all, kept_domain_all):
    cat = cat_map_all[name]
    static_dict = {f"dom_{d}": [1.0 if d == dom else 0.0] for d in ALL_DOMAINS}
    static_dict.update({f"cat_{c}": [1.0 if c == cat else 0.0] for c in ALL_CATEGORIES})
    static_df = pd.DataFrame(static_dict)

    month_cov = datetime_attribute_timeseries(cov.time_index, attribute="month", cyclic=True)
    cov_feat = cov.stack(month_cov)

    series_scaled_all_feat.append(s_scaled.with_static_covariates(static_df))
    train_target_all_feat.append(t_target.with_static_covariates(static_df))
    cov_scaled_all_feat.append(cov_feat)

print(f"static covariate 차원: 도메인{len(ALL_DOMAINS)} + 카테고리{len(ALL_CATEGORIES)} = {len(ALL_DOMAINS)+len(ALL_CATEGORIES)}")
print(f"past covariate 차원: {cov_scaled_all_feat[0].n_components} (리뷰수 1 + month_sin/cos 2)")

static covariate 차원: 도메인5 + 카테고리20 = 25
past covariate 차원: 3 (리뷰수 1 + month_sin/cos 2)


In [25]:
nhits_global2 = NHiTSModel(input_chunk_length=INPUT_LEN, output_chunk_length=OUTPUT_LEN,
                            n_epochs=75, random_state=42,
                            likelihood=QuantileRegression(quantiles=[0.1, 0.5, 0.9]),
                            pl_trainer_kwargs={"enable_progress_bar": False, "enable_model_summary": False, "accelerator": "cpu"})
nhits_global2.fit(series=train_target_all_feat, past_covariates=cov_scaled_all_feat)
print("피처보강 전역학습(도메인+카테고리+월계절성) 완료 --", len(train_target_all_feat), "개 시리즈")

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pytorch_lightning/utilities/data.py:106: Total length of `list` across ranks is zero. Please make sure this was your intention.
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


`Trainer.fit` stopped: `max_epochs=75` reached.


피처보강 전역학습(도메인+카테고리+월계절성) 완료 -- 1156 개 시리즈


In [26]:
fashion_series_scaled_all_feat = [series_scaled_all_feat[i] for i in fashion_idx_in_all]
fashion_cov_scaled_all_feat = [cov_scaled_all_feat[i] for i in fashion_idx_in_all]

nhits_global2_hist = nhits_global2.historical_forecasts(
    series=fashion_series_scaled_all_feat, past_covariates=fashion_cov_scaled_all_feat,
    forecast_horizon=OUTPUT_LEN, stride=STRIDE, start=TRAIN_FRAC,
    retrain=False, last_points_only=False, verbose=False, num_samples=200,
)
print("피처보강 전역학습 모델, 패션 상품 백테스트 완료")

GPU available: True (mps), used: False


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


피처보강 전역학습 모델, 패션 상품 백테스트 완료


In [27]:
nhits_global2_errs, naive_global2_errs, n_windows_global2 = [], [], 0
for i, nh_windows in enumerate(nhits_global2_hist):
    full_pd = fashion_series_all[i].to_series()
    for nh_w in nh_windows:
        actual = full_pd.reindex(nh_w.time_index).values
        nh_pred = nh_w.quantile(0.5).values().ravel() * (fashion_train_maxs_all[i] - fashion_train_mins_all[i]) + fashion_train_mins_all[i]

        naive_origin = nh_w.time_index[0] - pd.DateOffset(months=1)
        naive_val = full_pd.get(naive_origin, np.nan)
        if pd.isna(naive_val):
            continue

        n_windows_global2 += 1
        nhits_global2_errs.append(np.abs(actual - nh_pred).mean())
        naive_global2_errs.append(np.abs(actual - naive_val).mean())

nhits_global2_errs, naive_global2_errs = map(np.array, (nhits_global2_errs, naive_global2_errs))
print(f"윈도우 {n_windows_global2}개 (섹션9와 동일한 293개여야 정상)")
print()
print(f"naive 평균 MAE:                      {naive_global2_errs.mean():.4f}")
print(f"NHiTS(5도메인 전역학습, 섹션9) MAE:    {nhits_global_errs.mean():.4f}")
print(f"NHiTS(전역학습+피처보강) MAE:          {nhits_global2_errs.mean():.4f}")
print()
print(f"피처보강이 섹션9 모델보다 나은 윈도우: {(nhits_global2_errs < nhits_global_errs).mean()*100:.1f}%")
print(f"피처보강이 naive보다 나은 윈도우:      {(nhits_global2_errs < naive_global2_errs).mean()*100:.1f}%")

윈도우 293개 (섹션9와 동일한 293개여야 정상)

naive 평균 MAE:                      0.1406
NHiTS(5도메인 전역학습, 섹션9) MAE:    0.1330
NHiTS(전역학습+피처보강) MAE:          0.1268

피처보강이 섹션9 모델보다 나은 윈도우: 56.7%
피처보강이 naive보다 나은 윈도우:      45.4%


## 12. 조기탐지 후보 재계산 (5차 최종모델 기준)

섹션10의 조기탐지 후보는 4차(도메인통합, 피처보강 전) 모델 기준이었다. 실제 최종 채택 모델인
5차(`nhits_global2`, 도메인통합+카테고리+계절성)로 다시 계산한다. 로직은 섹션10과 동일.

In [28]:
MIN_ALERT_REVIEWS = 4

alerts_v4, alerts_v4_noisy = [], []
for i, (name, nh_windows) in enumerate(zip(fashion_names_all, nhits_global2_hist)):
    full_pd = fashion_series_all[i].to_series()
    full_counts = fashion_count_series_all[i].to_series()
    cut = int(len(fashion_series_all[i]) * TRAIN_FRAC)
    hist_vals = fashion_series_all[i].values().ravel()[:cut]
    own_max = hist_vals.max()
    own_mean, own_std = hist_vals.mean(), hist_vals.std()

    for nh_w in nh_windows:
        hi = nh_w.quantile(0.9).values().ravel()
        actual = full_pd.reindex(nh_w.time_index).values
        rev_counts = full_counts.reindex(nh_w.time_index).values

        for t, a, h, rc in zip(nh_w.time_index, actual, hi, rev_counts):
            h_real = h * (fashion_train_maxs_all[i] - fashion_train_mins_all[i]) + fashion_train_mins_all[i]
            if a <= h_real:
                continue
            row = {
                "product": name.replace("패션||", ""), "month": t, "actual": a, "upper_bound_real": h_real,
                "review_count": int(rc),
                "상품자체_역대최고치_초과": a > own_max,
                "상품자체_평균+1.5표준편차_초과": a > own_mean + 1.5 * own_std,
            }
            (alerts_v4 if rc >= MIN_ALERT_REVIEWS else alerts_v4_noisy).append(row)

alerts_v4_df = pd.DataFrame(alerts_v4)
alerts_v4_noisy_df = pd.DataFrame(alerts_v4_noisy)
print(f"[5차 최종모델] 1차 후보(90% 예측구간 상한 초과): 신뢰 가능(리뷰 {MIN_ALERT_REVIEWS}건 이상) {len(alerts_v4_df)}건, "
      f"노이즈로 제외 {len(alerts_v4_noisy_df)}건")
if len(alerts_v4_df):
    n_new_high = alerts_v4_df["상품자체_역대최고치_초과"].sum()
    print(f"그 중 '상품 자체 역대 최고치'보다도 높은 진짜 신규 급증: {n_new_high}건 ({n_new_high/len(alerts_v4_df)*100:.0f}%)")
print()
print(f"(참고, 섹션10 4차 모델 기준: 신뢰가능 53건, 진짜신규급증 10건/19%)")

[5차 최종모델] 1차 후보(90% 예측구간 상한 초과): 신뢰 가능(리뷰 4건 이상) 47건, 노이즈로 제외 86건
그 중 '상품 자체 역대 최고치'보다도 높은 진짜 신규 급증: 7건 (15%)

(참고, 섹션10 4차 모델 기준: 신뢰가능 53건, 진짜신규급증 10건/19%)


In [29]:
alerts_v4_df.sort_values("actual", ascending=False).head(10)

,product,month,actual,upper_bound_real,review_count,상품자체_역대최고치_초과,상품자체_평균+1.5표준편차_초과
20,OO 볼** 소가죽 여성 캐주얼부츠,2019-12-01,1.000000,0.000344,5,True,True
17,OO 보** 여성샌들 + 슬리퍼 2종,2021-07-01,0.800000,0.084597,5,True,True
39,OO 여성 썸머 시** 팬츠 3종,2022-02-01,0.600000,0.207449,5,False,False
25,OO 여름 에** 재킷 2종,2021-06-01,0.571429,0.180037,7,False,True
2,OO 남성 봄 풀오버 니트 3종,2021-09-01,0.500000,0.067113,4,False,True
3,OO 남성 봄 풀오버 니트 3종,2021-10-01,0.500000,0.132670,4,False,True
26,OO 여성 기모 본딩팬츠 3종,2021-11-01,0.500000,0.133933,4,True,True
16,OO 보** 여성샌들 + 슬리퍼 2종,2021-06-01,0.500000,0.217929,4,True,True
0,OO 남성 더** 데님 3종,2022-01-01,0.400000,0.300093,10,True,True
12,OO 남성 풀** 데님팬츠 3종,2021-12-01,0.384615,0.114419,13,False,False
